In [22]:
# Instalar a biblioteca oficial do Google GenAI e ferramentas de análise
!pip install -q -U google-genai pandas scikit-learn

import os
import json
import pandas as pd
from google import genai
from google.genai import types

# Insira sua chave da API do Google AI Studio (://google.com)
CHAVE_API = "SUA_CHAVE_API_AQUI"



client = genai.Client(api_key=CHAVE_API)
MODELO = "gemini-3.8-flash" # Modelo rápido e econômico para triagem
print("Ambiente configurado com sucesso!")


Ambiente configurado com sucesso!


In [4]:
# Criando dataset sintético representando as 7 categorias e casos críticos
dados_estudantes = [
    {"id": 1, "mensagem": "Oi, não consigo entrar no AVA, diz senha inválida. Meu CPF é 123.456.789-00.", "assunto_real": "acesso ao ambiente virtual", "prioridade_real": "alta", "sensivel": True},
    {"id": 2, "mensagem": "Quero saber quando começam as provas finais deste semestre.", "assunto_real": "calendário acadêmico", "prioridade_real": "baixa", "sensivel": False},
    {"id": 3, "mensagem": "Paguei o boleto ontem via Pix mas ainda consta como pendente no portal. Segue comprovante.", "assunto_real": "pagamentos", "prioridade_real": "media", "sensivel": True},
    {"id": 4, "mensagem": "O professor corrigiu errado a questão 3 da prova de Cálculo! Preciso de revisão de nota urgente, senão perco minha bolsa ProUni.", "assunto_real": "revisão de notas", "prioridade_real": "alta", "sensivel": True},
    {"id": 5, "mensagem": "Preciso do meu histórico escolar assinado digitalmente para apresentar no estágio até sexta.", "assunto_real": "documentos", "prioridade_real": "media", "sensivel": True},
    {"id": 6, "mensagem": "Como faço para trancar minha matrícula? Estou passando por problemas graves de saúde e depressão.", "assunto_real": "matrícula", "prioridade_real": "alta", "sensivel": True},
    {"id": 7, "mensagem": "Gostaria de solicitar aproveitamento de disciplinas cursadas em outra faculdade.", "assunto_real": "solicitações administrativas", "prioridade_real": "baixa", "sensivel": False},
    {"id": 8, "mensagem": "Meu boleto venceu e não consigo gerar segunda via, mas também quero saber se tenho desconto por pontualidade.", "assunto_real": "pagamentos", "prioridade_real": "media", "sensivel": True}
]

df = pd.DataFrame(dados_estudantes)
display(df)


,id,mensagem,assunto_real,prioridade_real,sensivel
0,1,"Oi, não consigo entrar no AVA, diz senha invál...",acesso ao ambiente virtual,alta,True
1,2,Quero saber quando começam as provas finais de...,calendário acadêmico,baixa,False
2,3,Paguei o boleto ontem via Pix mas ainda consta...,pagamentos,media,True
3,4,O professor corrigiu errado a questão 3 da pro...,revisão de notas,alta,True
4,5,Preciso do meu histórico escolar assinado digi...,documentos,media,True
5,6,Como faço para trancar minha matrícula? Estou ...,matrícula,alta,True
6,7,Gostaria de solicitar aproveitamento de discip...,solicitações administrativas,baixa,False
7,8,Meu boleto venceu e não consigo gerar segunda ...,pagamentos,media,True


In [5]:
def classificador_regras_simples(texto):
    texto = texto.lower()
    if any(p in texto for p in ["senha", "ava", "login", "acesso"]):
        return "acesso ao ambiente virtual"
    elif any(p in texto for p in ["boleto", "pix", "paguei", "pagamento", "desconto"]):
        return "pagamentos"
    elif any(p in texto for p in ["prova", "calendário", "data", "semestre"]):
        return "calendário acadêmico"
    elif any(p in texto for p in ["nota", "revisão", "corrigiu"]):
        return "revisão de notas"
    elif any(p in texto for p in ["histórico", "declaração", "documento"]):
        return "documentos"
    elif any(p in texto for p in ["matrícula", "trancar"]):
        return "matrícula"
    else:
        return "solicitações administrativas"

# Testando o classificador tradicional
df["predicao_regras"] = df["mensagem"].apply(classificador_regras_simples)
acuracia_regras = (df["assunto_real"] == df["predicao_regras"]).mean() * 100

print(f"Acurácia do método tradicional (Palavras-chave): {acuracia_regras:.1f}%")
display(df[["id", "mensagem", "assunto_real", "predicao_regras"]])


Acurácia do método tradicional (Palavras-chave): 87.5%


,id,mensagem,assunto_real,predicao_regras
0,1,"Oi, não consigo entrar no AVA, diz senha invál...",acesso ao ambiente virtual,acesso ao ambiente virtual
1,2,Quero saber quando começam as provas finais de...,calendário acadêmico,calendário acadêmico
2,3,Paguei o boleto ontem via Pix mas ainda consta...,pagamentos,pagamentos
3,4,O professor corrigiu errado a questão 3 da pro...,revisão de notas,calendário acadêmico
4,5,Preciso do meu histórico escolar assinado digi...,documentos,documentos
5,6,Como faço para trancar minha matrícula? Estou ...,matrícula,matrícula
6,7,Gostaria de solicitar aproveitamento de discip...,solicitações administrativas,solicitações administrativas
7,8,Meu boleto venceu e não consigo gerar segunda ...,pagamentos,pagamentos


In [25]:
# Prompt 1: Ingênuo (Sem restrições de formato ou política)
def testar_prompt_ingenuo(mensagem):
    prompt = f"Classifique esta mensagem de aluno, diga a prioridade e escreva uma resposta: {mensagem}"
    response = client.models.generate_content(model=MODELO, contents=prompt)
    return response.text

# Prompt 2: Estruturado com Guardrails (Saída JSON controlada)
def testar_prompt_estruturado(mensagem):
    system_instruction = """
    Você é um assistente de triagem de uma instituição de ensino.
    Analise a mensagem e retorne EXCLUSIVAMENTE um JSON válido com as chaves:
    - "assunto": escolha entre [matrícula, documentos, pagamentos, calendário acadêmico, revisão de notas, acesso ao ambiente virtual, solicitações administrativas]
    - "prioridade": [baixa, media, alta] (Alta se houver prazo curto, bloqueio de acesso, risco de perda de bolsa ou sofrimento emocional)
    - "pode_responder_sozinho": booleano (True APENAS para dúvidas gerais de calendário ou procedimento padrão sem dados pessoais. False para notas, finanças, saúde ou documentos)
    - "rascunho_resposta": texto cordial, sem inventar datas, prazos ou valores. Se faltar informação, peça para aguardar o setor responsável.
    """

    response = client.models.generate_content(
        model=MODELO,
        contents=mensagem,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            response_mime_type="application/json",
            temperature=0.1 # Baixa temperatura reduz criatividade/alucinação
        )
    )
    return json.loads(response.text)

# Testando nos casos críticos (ID 4 - Revisão de nota e ID 6 - Saúde/Trancamento)
print("=== TESTE PROMPT INGÊNUO (CASO 4) ===")
print(testar_prompt_ingenuo(df.loc[3, "mensagem"]))

print("\n=== TESTE PROMPT ESTRUTURADO (CASO 4) ===")
print(json.dumps(testar_prompt_estruturado(df.loc[3, "mensagem"]), indent=2, ensure_ascii=False))


=== TESTE PROMPT INGÊNUO (CASO 4) ===
Aqui está a análise da mensagem e a proposta de resposta:

### 1. Classificação
* **Categoria:** Acadêmico / Pedido de Revisão de Prova (Nota).
* **Subcategoria:** Risco de Bolsa (ProUni) / Risco de Evasão.

---

### 2. Prioridade
* **Nível:** **ALTA (Urgente)**
* **Motivo:** Envolve um prazo acadêmico sensível (revisão de prova) com impacto direto e grave na permanência do estudante na instituição (perda de benefício financeiro/ProUni).

---

### 3. Sugestão de Resposta

> **Assunto/Mensagem:**
>
> Olá, [Nome do Aluno], compreendo perfeitamente a sua preocupação. Sabemos o quanto a bolsa do ProUni é importante para você e vamos te orientar para resolver essa situação o mais rápido possível.
>
> Para que a revisão da sua questão seja feita formalmente, por favor, siga o passo a passo abaixo:
>
> 1. **Abra o requerimento oficial:** Acesse o seu Portal do Aluno > Menu *Secretaria/Requerimentos* > selecione a opção **"Revisão de Nota/Prova"**.
> 2. **

In [29]:
import time

resultados_ia = []

for index, row in df.iterrows():
    sucesso = False
    tentativas = 0

    while not sucesso and tentativas < 3:
        try:
            res = testar_prompt_estruturado(row["mensagem"])
            resultados_ia.append(res)
            sucesso = True
            print(f"Mensagem {row['id']} processada com sucesso!")
            time.sleep(2) # Pausa de 2 segundos para evitar bloqueio por excesso de requisições
        except Exception as e:
            tentativas += 1
            print(f"Servidor ocupado na mensagem {row['id']}. Tentando novamente em 5s... (Tentativa {tentativas}/3)")
            time.sleep(5)

    # Caso o servidor continue fora do ar após 3 tentativas, aplica fallback seguro
    if not sucesso:
        resultados_ia.append({
            "assunto": "ERRO_API_INDISPONIVEL",
            "prioridade": "alta",
            "pode_responder_sozinho": False,
            "rascunho_resposta": "Encaminhar para triagem humana (Fallback de indisponibilidade)."
        })

df_ia = pd.DataFrame(resultados_ia)
df_final = pd.concat([df, df_ia], axis=1)

acuracia_ia = (df_final["assunto_real"] == df_final["assunto"]).mean() * 100
print(f"\nAcurácia do LLM Estruturado: {acuracia_ia:.1f}%")
display(df_final[["id", "assunto_real", "assunto", "prioridade", "pode_responder_sozinho", "rascunho_resposta"]])


Servidor ocupado na mensagem 1. Tentando novamente em 5s... (Tentativa 1/3)
Servidor ocupado na mensagem 1. Tentando novamente em 5s... (Tentativa 2/3)
Mensagem 1 processada com sucesso!
Servidor ocupado na mensagem 2. Tentando novamente em 5s... (Tentativa 1/3)
Mensagem 2 processada com sucesso!
Mensagem 3 processada com sucesso!
Mensagem 4 processada com sucesso!
Mensagem 5 processada com sucesso!
Servidor ocupado na mensagem 6. Tentando novamente em 5s... (Tentativa 1/3)
Servidor ocupado na mensagem 6. Tentando novamente em 5s... (Tentativa 2/3)
Mensagem 6 processada com sucesso!
Servidor ocupado na mensagem 7. Tentando novamente em 5s... (Tentativa 1/3)
Servidor ocupado na mensagem 7. Tentando novamente em 5s... (Tentativa 2/3)
Mensagem 7 processada com sucesso!
Servidor ocupado na mensagem 8. Tentando novamente em 5s... (Tentativa 1/3)
Servidor ocupado na mensagem 8. Tentando novamente em 5s... (Tentativa 2/3)
Servidor ocupado na mensagem 8. Tentando novamente em 5s... (Tentativa 

,id,assunto_real,assunto,prioridade,pode_responder_sozinho,rascunho_resposta
0,1,acesso ao ambiente virtual,acesso ao ambiente virtual,alta,False,Olá! Lamentamos o inconveniente com o acesso a...
1,2,calendário acadêmico,calendário acadêmico,baixa,True,Olá! As datas das provas finais constam no cal...
2,3,pagamentos,pagamentos,media,False,Olá! Agradecemos pelo envio do comprovante. Co...
3,4,revisão de notas,revisão de notas,alta,False,Olá! Compreendemos a urgência da sua solicitaç...
4,5,documentos,documentos,alta,False,Olá! Compreendemos a urgência da sua solicitaç...
5,6,matrícula,matrícula,alta,False,Olá. Sentimos muito pelo momento difícil que v...
6,7,solicitações administrativas,solicitações administrativas,media,False,Olá! Para solicitar o aproveitamento de discip...
7,8,pagamentos,ERRO_API_INDISPONIVEL,alta,False,Encaminhar para triagem humana (Fallback de in...


In [28]:
import re

def anonimizar_dados_sensiveis(texto):
    # Mascara CPFs no formato XXX.XXX.XXX-XX ou 11 dígitos
    texto = re.sub(r'\d{3}\.\d{3}\.\d{3}-\d{2}|\b\d{11}\b', '[CPF_OCULTO]', texto)
    # Mascara possíveis RAs / Matrículas de 6 a 10 dígitos
    texto = re.sub(r'\b\d{6,10}\b', '[MATRICULA_OCULTA]', texto)
    return texto

# Exemplo prático de governança de dados (Privacy by Design)
msg_original = df.loc[0, "mensagem"]
msg_segura = anonimizar_dados_sensiveis(msg_original)

print("Original (Risco LGPD):", msg_original)
print("Anonimizada (Segura para API):", msg_segura)


Original (Risco LGPD): Oi, não consigo entrar no AVA, diz senha inválida. Meu CPF é 123.456.789-00.
Anonimizada (Segura para API): Oi, não consigo entrar no AVA, diz senha inválida. Meu CPF é [CPF_OCULTO].


# Parecer Técnico-Executivo: Análise de Viabilidade de IA no Atendimento Estudantil

## 1. Resposta à Pergunta Motriz
**"Esse problema deveria realmente ser resolvido com inteligência artificial?"**

**Não integralmente.** A recomendação técnica é adotar uma **arquitetura híbrida (Automação Convencional + IA como Copiloto + Supervisão Humana)**, e **não** um sistema 100% autônomo baseado em Modelos de Linguagem (LLMs).

Os experimentos realizados demonstraram que o uso irrestrito de IA traz riscos legais, operacionais e de confiabilidade que superam os ganhos de velocidade em tarefas críticas.

---

## 2. Consolidação das Evidências por Nível

### Nível Básico: Decomposição e Comparação com Métodos Simples
* **Onde regras convencionais bastam:** O encaminhamento da mensagem após a identificação do tema é uma tarefa puramente determinística (`De -> Para`). Utilizar um LLM para decidir para qual fila enviar uma solicitação já classificada gera custo desnecessário e latência.
* **Onde regras falham e a IA agrega valor:** Classificadores baseados apenas em palavras-chave falham diante de mensagens longas, ambíguas ou com múltiplos temas (ex.: um aluno que menciona *"prova"* ao pedir *"revisão de nota"*). O LLM demonstrou superioridade na compreensão semântica do contexto real da solicitação.

### Nível Intermediário: Engenharia de Prompts e Qualidade
* **Risco do Prompt Ingênuo:** Prompts abertos geram respostas prolixas, tom inconsistente e risco de alucinação (invenção de prazos ou procedimentos acadêmicos).
* **Ganho com Prompt Estruturado (JSON + Baixa Temperatura):** Ao restringir a saída para um esquema JSON fechado, definir temperatura baixa (`0.1`) e aplicar travas de segurança (`pode_responder_sozinho: False` para temas sensíveis), o modelo tornou-se previsível e seguro para atuar como ferramenta de triagem e apoio à redação.

### Nível Avançado: Riscos Éticos, Legais e Governança
* **Privacidade e LGPD:** As mensagens contêm dados pessoais e sensíveis (CPF, matrícula, situação financeira e relatos de saúde mental). Enviar esses dados diretamente para APIs externas sem tratamento viola o princípio da minimização de dados da LGPD. A implementação de uma camada local de anonimização (*PII Masking* via Regex) mostrou-se indispensável antes de qualquer chamada ao modelo.
* **Indisponibilidade Operacional (Erro 503):** Durante os testes em lote, a API externa apresentou falha temporária de sobrecarga (`503 UNAVAILABLE`). Isso comprova que a operação da instituição não pode depender exclusivamente de serviços de terceiros sem um mecanismo de *fallback* (contingência) local.
* **Proibição de Decisões Automatizadas Críticas:** Conforme o Art. 20 da LGPD e princípios éticos educacionais, decisões sobre **revisão de notas, concessão de bolsas/benefícios e medidas disciplinares** não devem ser tomadas por IA, devido ao risco de viés algorítmico e impacto direto na vida acadêmica do estudante.

---

## 3. Matriz de Recomendação Final por Funcionalidade

| Atividade Analisada | Recomendação | Tecnologia / Abordagem | Condição de Governança |
| :--- | :--- | :--- | :--- |
| **1. Classificar assunto** | **Usar IA** | LLM com saída JSON estruturada | Exigir anonimização prévia (LGPD) e fila de *fallback* se a API cair. |
| **2. Identificar prioridade** | **Híbrido** | Regras críticas + LLM | Palavras-chave de urgência (saúde, prazo hoje, bolsa) forçam prioridade alta. |
| **3. Encaminhar ao setor** | **Não usar IA** | Lógica condicional (`If/Else`) | Roteamento determinístico baseado na categoria validada. |
| **4. Produzir rascunho** | **Usar IA (Copiloto)** | LLM conectado à base oficial | O texto serve apenas de apoio; revisão humana obrigatória antes do envio. |
| **5. Envio automático simples** | **Restrito** | FAQ / Fluxo guiado ou LLM travado | Liberado **apenas** para dúvidas públicas gerais (ex.: datas do calendário). |
| **6. Decisões (Notas/Bolsas)** | **Proibido usar IA** | Processo 100% humano | Decisão exclusiva da equipe acadêmica/financeira com registro auditável. |

---

## 4. Plano de Implantação Recomendado (Próximos Passos)
Caso a direção aprove o projeto sob estas condições, recomenda-se iniciar por uma **Fase Piloto (Shadow Mode)** de 30 dias:
1. A IA classifica as mensagens (já anonimizadas) e sugere o rascunho internamente no sistema de chamados.
2. Nenhum e-mail é enviado automaticamente ao aluno nessa etapa.
3. Os atendentes avaliam a qualidade das sugestões para calibrar as métricas de precisão antes de liberar qualquer automação de resposta simples.
